# Decision Trees (Classification & Regression)

**Decision Trees** are one of the most popular, intuitive, and versatile machine learning algorithms. They are **non-parametric** supervised learning models used for both **classification** (predicting categorical classes) and **regression** (predicting continuous numerical values). They work by recursively partitioning the feature space into homogeneous sub-regions based on hierarchical decision rules.

In these detailed notes, we will cover:
1. **Core Intuition & Logic:** Nested if-else statements and the anatomy of a tree.
2. **Geometric Interpretation:** Axis-aligned decision boundaries and hyper-cuboids.
3. **Key Metrics for Splitting (Classification):** Entropy and Information Gain.
4. **Gini Impurity:** Definition, mathematical properties, and comparison with Entropy (CART algorithm).
5. **Handling Continuous/Numerical Data:** Threshold selection via sorting and evaluation.
6. **Hyperparameter Tuning:** Managing model complexity to prevent overfitting and underfitting.
7. **Regression Trees:** Mechanics of continuous predictions, error minimization (MSE/MAE), and multiple features.
8. **Feature Importance:** How trees measure feature relevance for selection and simplification.
9. **Practical Python Demos:**
   - **Demo A:** Classification boundary visualization (Overfitted vs. Regularized).
   - **Demo B:** Hyperparameter tuning using `GridSearchCV` on the Breast Cancer dataset.
   - **Demo C:** Regression Tree fitting on non-linear data (visualizing step-wise predictions).
   - **Demo D:** Feature Importance analysis.
   - **Summary Checklist** for Decision Trees.

## 1. Intuition and Logic

### The Core Concept
At their core, Decision Trees are essentially **nested if-else statements**. They perform a sequence of binary splits on the features of the dataset to categorize outcomes.

### Structure of a Decision Tree
A decision tree consists of three primary types of nodes:
*   **Root Node (Top):** The starting point of the tree containing the entire dataset. It represents the first feature evaluation and split.
*   **Decision Nodes (Internal Splits):** Sub-nodes that split into further sub-nodes based on specific feature conditions (e.g., $X_j \le \theta$).
*   **Leaf Nodes (Terminal Nodes):** The endpoints of the tree that do not split further. They contain the final output value or predicted class.

```
          [ Root Node ]
             /     \
         Split 1  Split 2
           /         \
    [Leaf Node]    [Decision Node]
                      /       \
                  Split 3   Split 4
                    /           \
              [Leaf Node]   [Leaf Node]
```

### Illustrative Example: App Downloads Prediction
Consider a Play Store dataset where we want to predict whether a user will download a specific app based on demographic features:
1.  **Root Node:** *"Is the user a student?"*
    *   **Yes:** Go to next evaluation: *"Is the gender female?"*
        *   **Yes:** **Leaf Node** -> Predict *"Downloads App"*.
        *   **No:** **Leaf Node** -> Predict *"Does Not Download"*.
    *   **No:** Go to next evaluation: *"Is the user's age < 25?"*
        *   **Yes:** **Leaf Node** -> Predict *"Downloads App"*.
        *   **No:** **Leaf Node** -> Predict *"Does Not Download"*.

This nested binary questioning mirrors human decision-making, making the algorithm highly explainable.

## 2. Geometric Interpretation

### Axis-Aligned Decision Boundaries
Mathematically, decision trees partition the feature space into distinct regions. 
*   In **2D space** (two features), a split corresponds to drawing a straight line **parallel to one of the coordinate axes** (e.g., $x_1 = c$ or $x_2 = d$).
*   In **higher dimensions** (e.g., 3D or more), the splits correspond to orthogonal planes or hyperplanes, partitioning the space into **rectangles** (in 2D) or **hyper-cuboids** (in $d$-dimensions).

### Recursive Space Partitioning
Unlike algorithms like Logistic Regression or SVMs that draw a single diagonal decision boundary (linear split), a decision tree recursively cuts the coordinate system into finer grid-like cells. The tree continues cutting the coordinate system recursively until the stopping criteria are met or the data is classified.

## 3. Key Metrics for Splitting (Purity & Entropy)

To build a tree, the algorithm must decide **which feature** to split on and **at what threshold**. It chooses splits that maximize node **purity** (homogeneity).

### 1. Entropy
Entropy is a measure of the **disorder, impurity, or randomness** in a dataset. Lower entropy indicates higher purity (more homogeneous classes), while higher entropy indicates greater impurity.

For a dataset $S$ with $c$ classes, the Entropy $H(S)$ is defined as:

$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

Where:
*   $p_i$ is the probability or proportion of samples belonging to class $i$ in dataset $S$.
*   $\log_2$ is the base-2 logarithm.

#### Key Values of Entropy:
*   **$H(S) = 0$ (Minimum Impurity / Pure):** All samples in the node belong to a single class (e.g., only positive samples).
*   **$H(S) = 1$ (Maximum Impurity / Impure):** For a binary class setting, samples are evenly split 50/50 between the two classes.

### 2. Information Gain
Information Gain (IG) measures the **reduction in entropy** achieved after partitioning a dataset based on a given feature.

$$IG(S, A) = H(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

Where:
*   $H(S)$ is the entropy of the parent node.
*   $A$ is the feature being evaluated for the split.
*   $\text{Values}(A)$ are the subsets created by the split.
*   $S_v$ is the subset of $S$ where feature $A$ takes value $v$.
*   $\frac{|S_v|}{|S|}$ represents the weight of the subset (proportion of samples in the child node relative to the parent).

**Decision Rule:** The algorithm computes the Information Gain for all possible splits and selects the split that yields the **maximum Information Gain**.

## 4. Gini Impurity

An alternative splitting metric to Entropy is **Gini Impurity**. This is the default metric used by Scikit-Learn's `DecisionTreeClassifier` (via the **CART** - Classification and Regression Trees algorithm).

### Definition
Gini Impurity measures the probability of misclassifying a randomly chosen element from the dataset if it were randomly labeled according to the class distribution of the node.

The Gini Impurity of a dataset $S$ is defined as:

$$Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$$

Where $p_i$ is the probability/proportion of samples belonging to class $i$.

#### Key Values of Gini Impurity:
*   **$Gini(S) = 0$ (Pure Node):** All samples belong to a single class.
*   **$Gini(S) = 0.5$ (Binary Maximum Impurity):** For a binary classification, the classes are split 50/50.

### Gini vs. Entropy: The Benefit of Gini
1.  **Mathematical Form:** Entropy uses logarithmic functions ($p_i \log_2 p_i$), whereas Gini uses simple squaring ($p_i^2$).
2.  **Computational Efficiency:** Calculating logarithms is computationally intensive. Gini Impurity avoids log operations, making it **much faster** to compute during tree construction.
3.  **Behavior:** Gini and Entropy yield similar trees in $90\%$ of cases. Gini tends to isolate the most frequent class in its own branch, while Entropy tends to produce more balanced splits.

## 5. Handling Continuous/Numerical Data

Decision Trees natively handle categorical features, but they must use a specific thresholding mechanism for continuous variables.

### The Algorithm for Continuous Split Finding:
1.  **Sort the Data:** For a continuous feature $X_j$ (e.g., age, income), sort the unique values in ascending order.
2.  **Generate Candidate Thresholds:** Calculate the midpoints between every adjacent pair of sorted values. These serve as potential splitting points ($	heta$).
3.  **Evaluate Splits:** For each candidate threshold $	heta$, split the dataset into two groups:
    *   $S_{\text{left}} = \{x \mid x_j \le \theta\}$
    *   $S_{\text{right}} = \{x \mid x_j > \theta\}$
4.  **Calculate Metric:** Compute the Information Gain (or Gini reduction) for each split.
5.  **Select Best Split:** The threshold $	heta$ that maximizes the metric is chosen as the splitting condition for that feature.

This allows decision trees to naturally handle numerical variables without requiring manual discretization or binning.

## 6. Advantages & Disadvantages of Decision Trees

| Advantages (Pros) | Disadvantages (Cons) |
| :--- | :--- |
| **Highly Intuitive:** White-box model; easy to visualize and interpret. | **Prone to Overfitting:** Unconstrained trees grow deep and fit training noise perfectly. |
| **No Feature Scaling Required:** Scales are irrelevant because splits happen on single-feature thresholds. | **High Variance:** Small modifications in training data can lead to a completely different tree. |
| **Handles Mixed Data Types:** Out-of-the-box support for both numerical and categorical inputs. | **Imbalanced Datasets:** Splits favor dominant classes to greedily reduce impurity. |
| **Fast Inference Time:** Prediction time is $O(\log n)$ where $n$ is tree depth. | **Axis-Aligned Splits:** Hard to fit diagonal relationships; creates step-like approximations. |

## 7. Hyperparameter Tuning & Model Complexity

### Overfitting vs. Underfitting
*   **Overfitting:** Occurs when a decision tree grows too deep (`max_depth` is too high). It learns hyper-specific, noisy rules, achieving $100\%$ training accuracy but failing to generalize to unseen test data (high variance).
*   **Underfitting:** Occurs when the tree is too simple or shallow (`max_depth` is too low). It fails to capture the underlying patterns, yielding poor accuracy on both training and test sets (high bias).
*   **The Tuning Goal:** Find the optimal combination of hyperparameters to restrict tree growth and find the "sweet spot" of generalization.

### Crucial Hyperparameters in Scikit-Learn
*   **`criterion`:** The metric used to evaluate splits (`'gini'` or `'entropy'`). Gini is generally preferred for computational performance.
*   **`splitter`:** Defines the strategy to choose the split at each node. `'best'` evaluates all splits; `'random'` selects a random split (which helps reduce overfitting/variance).
*   **`max_depth`:** The maximum depth of the tree. Restricting this is the most direct way to control overfitting.
*   **`min_samples_split`:** The minimum number of samples required to split an internal node. Higher values prevent the model from learning overly specific rules.
*   **`min_samples_leaf`:** The minimum number of samples required to exist at a leaf node. Smooths the model and prevents overfitting.
*   **`max_features`:** The number of features to consider when looking for the best split. Reducing this can help reduce overfitting in high-dimensional datasets.
*   **`min_impurity_decrease`:** A node will be split if this split induces a decrease of the impurity greater than or equal to this value.

### Practical Tools
For practical parameter validation, interactive visualization tools (like `dt-visualise`) let you visually observe how changing parameters changes the boundary shapes.

## 8. Regression Trees

While standard Decision Trees are used for classification, **Regression Trees** are designed to predict continuous numerical values.

### Non-Linear Relationships
Unlike Linear Regression, which assumes a straight-line relationship, Regression Trees can capture complex, non-linear patterns in data. For example, in predicting exam marks based on study hours, if the trend is non-linear, a regression tree creates splits (thresholds) to partition the data into segments and predicts the mean value of each segment.

### Splitting Mechanism
Instead of Gini Impurity or Entropy, Regression Trees use error metrics to evaluate split quality. The default metric in Scikit-Learn is **Mean Squared Error (MSE)**.

For a candidate split that partitions the dataset into two regions $R_1$ and $R_2$:
1.  **Region Prediction:** The predicted value for any point falling into a region is the **mean** of the targets of all training points in that region.
    $$\hat{y}_{R_1} = \frac{1}{N_1} \sum_{i \in R_1} y_i, \quad \hat{y}_{R_2} = \frac{1}{N_2} \sum_{i \in R_2} y_i$$
2.  **Error Calculation:** The algorithm measures the performance of a split by comparing the actual target values in the resulting groups to their mean (e.g., minimizing the sum of squared errors):
    $$SSE = \sum_{i \in R_1} (y_i - \hat{y}_{R_1})^2 + \sum_{i \in R_2} (y_i - \hat{y}_{R_2})^2$$
3.  **Optimal Split:** The algorithm chooses the split that minimizes the overall error. This is repeated recursively until a stopping criterion is met.

### Handling Multiple Features
When multiple features (e.g., Hours Studied and CGPA) are involved, the tree evaluates potential splits for each feature sequentially. It decides which feature to split on first based on which one results in the most effective error reduction.

## 9. Feature Importance

Decision Trees provide a built-in attribute, `feature_importances_`, which helps identify which input features have the most influence on the target variable.

### How it is calculated:
1.  For each node split, the reduction in impurity (Gini/Entropy for classification, MSE/MAE for regression) is computed, weighted by the fraction of training samples passing through that node:
    $$\Delta I = N_{\text{parent}} \times \text{Impurity}_{\text{parent}} - (N_{\text{left}} \times \text{Impurity}_{\text{left}} + N_{\text{right}} \times \text{Impurity}_{\text{right}})$$
2.  The importances are summed for each feature across all nodes split on that feature.
3.  The values are normalized so that the total sum of importances is $1.0$.

### Key Applications:
*   **Feature Selection:** Identifying and removing irrelevant columns from the dataset.
*   **Model Simplification:** Focus the model on the most impactful data points to speed up execution and enhance explainability.

## 10. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 11. Demo A: Decision Tree Classifier (Visualizing Boundaries & Overfitting)

We will generate a synthetic 2D classification dataset to visualize the decision boundaries of:
1.  An **unconstrained Decision Tree** (prone to overfitting).
2.  A **regularized Decision Tree** (with restricted depth and leaf size).

In [ ]:
# Generate synthetic 2D dataset
X_clf, y_clf = make_classification(
    n_samples=150, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.0, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# Train two models: Unconstrained (overfitting) and Constrained
clf_overfit = DecisionTreeClassifier(random_state=42)
clf_overfit.fit(X_train, y_train)

clf_regularized = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
clf_regularized.fit(X_train, y_train)

# Helper function to plot decision boundaries
def plot_decision_boundary(clf, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=50, label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#e74c3c', edgecolor='k', s=50, label='Class 1')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(frameon=True, facecolor='white')

# Plot boundaries
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_decision_boundary(clf_overfit, X_train, y_train, axes[0], f"Overfitted Tree (No Limits)\nTrain Acc: {clf_overfit.score(X_train, y_train):.4f} | Test Acc: {clf_overfit.score(X_test, y_test):.4f}")
plot_decision_boundary(clf_regularized, X_train, y_train, axes[1], f"Regularized Tree (max_depth=3, min_samples_leaf=5)\nTrain Acc: {clf_regularized.score(X_train, y_train):.4f} | Test Acc: {clf_regularized.score(X_test, y_test):.4f}")
plt.tight_layout()
plt.show()

# Visualize the tree structure of the regularized model
plt.figure(figsize=(14, 8))
plot_tree(clf_regularized, filled=True, feature_names=['Feature 1', 'Feature 2'], class_names=['Class 0', 'Class 1'], rounded=True)
plt.title("Visualizing the Regularized Decision Tree Structure", fontsize=14, fontweight='bold')
plt.show()

## 12. Demo B: Hyperparameter Tuning via GridSearchCV

We will demonstrate hyperparameter tuning on Scikit-Learn's built-in **Breast Cancer Wisconsin** dataset using `GridSearchCV`. This shows how to find the optimal parameter set that avoids overfitting and underfitting.

In [ ]:
# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names = cancer.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define parameter grid
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 4, 5, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 5, 10]
}

# Run Grid Search CV
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Evaluate the best model
best_clf = grid_search.best_estimator_
y_pred = best_clf.predict(X_test)

print("GridSearchCV Results:")
print("="*55)
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy:     {grid_search.best_score_:.4f}")
print(f"Test Set Accuracy:     {accuracy_score(y_test, y_pred):.4f}")
print("="*55)
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

## 13. Demo C: Regression Trees on Non-Linear Data

We will generate a synthetic non-linear 1D dataset (a sine wave with noise) and use `DecisionTreeRegressor` to fit different models. We will observe the characteristic step-wise, piecewise constant predictions of regression trees.

In [ ]:
# Generate synthetic non-linear dataset
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, X_reg.shape[0])

# Fit Regression Trees with different max_depth values
regr_1 = DecisionTreeRegressor(max_depth=2, random_state=42)
regr_2 = DecisionTreeRegressor(max_depth=5, random_state=42)
regr_unconstrained = DecisionTreeRegressor(random_state=42)

regr_1.fit(X_reg, y_reg)
regr_2.fit(X_reg, y_reg)
regr_unconstrained.fit(X_reg, y_reg)

# Predict
X_grid = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_1 = regr_1.predict(X_grid)
y_2 = regr_2.predict(X_grid)
y_unconstrained = regr_unconstrained.predict(X_grid)

# Plot predictions
plt.figure(figsize=(14, 7))
plt.scatter(X_reg, y_reg, s=40, edgecolor="black", c="darkorange", label="Data Points")
plt.plot(X_grid, y_1, color="cornflowerblue", label="max_depth=2 (Underfit/Simple)", linewidth=2.5)
plt.plot(X_grid, y_2, color="yellowgreen", label="max_depth=5 (Balanced)", linewidth=2.5)
plt.plot(X_grid, y_unconstrained, color="red", label="unconstrained (Overfit to Noise)", linewidth=1.5, linestyle="--")
plt.xlabel("Feature X", fontsize=12)
plt.ylabel("Target y", fontsize=12)
plt.title("Decision Tree Regression: Step-like Piecewise Constant Predictions", fontsize=14, fontweight="bold")
plt.legend(fontsize=12, frameon=True, facecolor="white")
plt.show()

# Print metrics
print(f"R² Score (max_depth=2): {r2_score(y_reg, regr_1.predict(X_reg)):.4f}")
print(f"R² Score (max_depth=5): {r2_score(y_reg, regr_2.predict(X_reg)):.4f}")
print(f"R² Score (unconstrained): {r2_score(y_reg, regr_unconstrained.predict(X_reg)):.4f} (Overfitting to training noise)")

## 14. Demo D: Visualizing Feature Importances

Using the tuned classifier from **Demo B**, we will plot and examine which features have the greatest influence on the decision tree's classification boundaries.

In [ ]:
# Extract importances
importances = best_clf.feature_importances_
indices = np.argsort(importances)[::-1]

# Create a DataFrame for visualization
feat_imp_df = pd.DataFrame({
    'Feature': np.array(feature_names)[indices],
    'Importance': importances[indices]
})

# Plot top 15 features
plt.figure(figsize=(12, 6))
sns.barplot(
    x='Importance', y='Feature', data=feat_imp_df.head(15),
    palette='viridis', edgecolor='black'
)
plt.title("Top 15 Feature Importances (Breast Cancer Dataset)", fontsize=14, fontweight='bold')
plt.xlabel("Normalized Impurity Reduction (Gini / Entropy)", fontsize=12)
plt.ylabel("Feature Name", fontsize=12)
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
print(feat_imp_df.head(5).to_string(index=False))

## Summary Checklist for Decision Trees

1.  **Understand Structure:** Root nodes split into decision nodes, ending at predicting leaf nodes.
2.  **Geometric Behavior:** Creates axis-aligned rectangular subdivisions in feature space (hyper-cuboids).
3.  **Choose Purity Metrics:** Use **Entropy** or **Gini Impurity** for classification. Gini is the default in CART and Scikit-Learn due to its computational speed (no log calculations).
4.  **No Feature Scaling Needed:** Decision Trees make decisions node-by-node based on thresholds. Unlike distance-based algorithms like KNN, scaling features does **not** affect their classification or splits.
5.  **Always Control Tree Growth:** Deep trees overfit (high variance). Control complexity using:
    *   `max_depth`: Limits max splits.
    *   `min_samples_split`: Require minimum parent size.
    *   `min_samples_leaf`: Guarantee minimum leaf size.
6.  **Continuous Data Handling:** Done automatically by sorting and evaluating intermediate threshold boundaries.
7.  **Regression Tree Target Prediction:** Leaf nodes predict the **mean** of the targets in that leaf region, producing step-like constant functions. Splits are selected to minimize **Mean Squared Error (MSE)**.
8.  **Feature Importance:** Tree models natively provide feature importances based on cumulative Gini/MSE reduction. Use `feature_importances_` to perform feature selection.
9.  **Struggles & Remedies:** Highly sensitive to training set perturbations. Remedy this by moving to ensemble methods like **Random Forests** or **Gradient Boosted Trees**.